# Validation B — cooling power, and where the paper breaks

**Akerboom *et al.*, *ACS Photonics* 9 (2022) 3831–3840,
[doi:10.1021/acsphotonics.2c01389](https://doi.org/10.1021/acsphotonics.2c01389)**

**No solver needed. This notebook runs in seconds.**

## The physics

A surface in sunlight settles where the powers balance:

$$P_\mathrm{cool}(T) = P_\mathrm{rad}(T) - P_\mathrm{atm} + P_\mathrm{conv}(T) - P_\mathrm{sun} = 0$$

$P_\mathrm{rad}$ is what the surface radiates, $\pi\int\langle\epsilon\rangle
B_\lambda(T)\,d\lambda$, rising steeply with temperature. $P_\mathrm{atm}$ is
what the sky radiates back, weighted by the atmosphere's own emissivity
$\epsilon_\mathrm{atm} = 1-\tau^{1/\cos\theta}$ — the 8–13 µm window is where
the sky is transparent and the surface can lose heat to space.
$P_\mathrm{conv} = h(T - T_\mathrm{amb})$ lumps all non-radiative exchange into
one coefficient. $P_\mathrm{sun}$ is the absorbed sunlight, here the paper's
stated 808 W/m².

This group takes the emittance from the paper's **measured** Figure 5a rather
than computing it, so it tests the thermal model *alone*. Any disagreement here
belongs to the energy balance, not to the optics.

## Main result — the paper does not reproduce itself

With the coefficient the paper's Methods state, $h = 6.0$ W/m²/K:

| Surface | radcoolpv | Paper Fig. 5b |
|---|---:|---:|
| Bare Au/Si | 415.4 K | 360 K |
| Flat silica | 360.6 K | 339 K |
| Silica cylinders | 355.6 K | 336 K |

A single fit to all three digitized curves gives $h = 12.54$ W/m²/K, which
reproduces 359.7 / 340.1 / 337.5 K — all three at once.

The inconsistency is decidable with arithmetic. Put a **perfect non-emitter**
under the paper's own balance: it radiates nothing, so

$$T = T_\mathrm{amb} + \frac{P_\mathrm{sun}}{h} = 300 + \frac{808}{6} = 434.7~\mathrm{K}.$$

Every real emitter must land *below* that. The paper's figure implies about
366.5 K for that limit, which no emitter can produce at $h = 6$.

**So the fitted agreement is a calibration, not an independent validation of
the thermal model, and must not be cited as one.**

## Set up the runtime

Colab runtimes are temporary. Run this again after a reset.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from IPython.display import Markdown, display

PROJECT = Path("/content/radcoolpv-py")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/gsilvaoelker/radcoolpv-py.git", str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--editable", "."],
               cwd=PROJECT, check=True)
os.chdir(PROJECT)

from radcoolpv import config, pipeline, report
print("radcoolpv ready in", PROJECT)

## The case

No geometry and no materials — just a spectrum, and the balance solved on it.
`optics: false` means S4 is never imported.

Switch `convection_coefficient` between `6.0` (the paper's stated value) and
`12.54` (the fitted one), and `optics_results_emittance_column` between 1, 2
and 3 for the three surfaces.

In [ ]:
%%writefile validation_b.yaml
run:
  optics: false                 # no solver: the emittance is measured data
  thermal: true
  plots: true
  mode: cooling_curve
  write_outputs: true
  results_dir: results/validation_b
  optics_results: validation/data/fig5a_measured_emittance.txt
  optics_results_angles: hemispherical
  optics_results_emittance_column: 3    # 1 bare, 2 flat silica, 3 cylinders

simulation:
  wavelength: {min: 2.0, max: 16.0, n: 281}
  angles: hemispherical

thermal:
  ambient_temperature: 300.0
  convection_coefficient: 6.0      # paper Methods; try 12.54, the fitted value
  absorbed_solar_power: 808.0      # paper eq. 2, absorbed not incident
  equilibrium: auto
  cooling_temperature: {min: 260.0, max: 440.0, n: 181}
  reference_curve_file: validation/data/fig5b_cooling_power.txt
  reference_curve_column: 3

In [ ]:
CASE = "validation_b.yaml"
ctx = pipeline.run(config.load_cases(CASE)[0])
report.summary(ctx)

## The zero-emitter check

No optics, no solver, no model — just the paper's own equation 2 at its stated coefficient.

In [ ]:
for h in (6.0, 12.15, 12.54):
    print(f"h = {h:5.2f} W/m2K  ->  perfect non-emitter settles at "
          f"{300.0 + 808.0 / h:6.2f} K")
print("\nThe paper's Fig. 5b implies ~366.5 K for that limit.")
print("No emitter can go above the non-emitter line, so h = 6.0 is inconsistent")
print("with the figure the same paper plots.")

**Exercise.** Sweep `convection_coefficient` from 6 to 15 W/m²/K and plot the equilibrium temperature against it. How much of the paper's claimed cooling survives if $h$ is uncertain by a factor of two?

## Use your own data

Upload a text file with wavelength (µm) in the first column and one or more
spectra in the others, then point `optics_results` at its filename and
`optics_results_emittance_column` at the column you want.

**If your file reaches below about 1.1 µm you also get the PV parameters.**
Above the band gap essentially everything absorbed is absorbed in the silicon,
so radcoolpv takes the silicon absorptance to equal the emittance there and
zero below; `run.json` records that this was assumed rather than solved.

A material works the same way: a CSV named `<Model>.csv` with columns
`lambda_um,n,k` becomes usable as `<Model>` in the `materials:` block.

In [ ]:
from google.colab import files

for name, blob in files.upload().items():
    head = blob[:200].decode(errors="ignore")
    if name.lower().endswith(".csv") and head.lower().startswith("lambda_um,"):
        (PROJECT / "radcoolpv" / "materials" / "data" / name).write_bytes(blob)
        print(f"{name}: installed as material {Path(name).stem!r}")
    else:
        print(f"{name}: {len(blob)} bytes, ready to use as optics_results")